In [ ]:
# Cell 1: Setup and Mount
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Ensure nibabel is installed for NIfTI processing
!pip install nibabel -q
print("Environment setup complete.")

Mounted at /content/drive
Environment setup complete.


In [ ]:
# Cell 2: Libraries and Configuration
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
# IMPORTANT: Update these paths to match where your shortcuts are in 'My Drive'
DRIVE_ROOT = "/content/drive/MyDrive/DNN asl"

DATA_ROOT = os.path.join(DRIVE_ROOT, "data (1)") # Shortcut to your 1CRTHm... folder
MODEL_DIR = os.path.join(DRIVE_ROOT, "model")  # Shortcut to your 1t0DuI... folder
OUTPUT_ROOT = os.path.join(DRIVE_ROOT, "RESULTS_SET2") # Shortcut to your 1ustcJ... folder

# Ensure output root exists
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print(f"Data directory: {DATA_ROOT}")
print(f"Model directory: {MODEL_DIR}")
print(f"Output directory: {OUTPUT_ROOT}")

Data directory: /content/drive/MyDrive/DNN asl/data (1)
Model directory: /content/drive/MyDrive/DNN asl/model
Output directory: /content/drive/MyDrive/DNN asl/RESULTS_SET2


In [ ]:
# Cell 3: Model Architecture
class BoundedNet(nn.Module):
    def __init__(self, input_dim, n_hidden_layers, n_neurons, y_min, y_max):
        super().__init__()
        self.y_min = y_min
        self.y_max = y_max
        layers = [nn.Linear(input_dim, n_neurons), nn.ELU()]
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.ELU()]
        layers.append(nn.Linear(n_neurons, 1))
        self.backbone = nn.Sequential(*layers)

    def forward(self, x):
        raw = self.backbone(x)
        return self.y_min + (self.y_max - self.y_min) * torch.sigmoid(raw)

print("Architecture defined.")

Architecture defined.


In [ ]:
# Cell 4: Load Weights and Stats
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize with exact parameters
CBFnet = BoundedNet(input_dim=4, n_hidden_layers=4, n_neurons=64, y_min=0.0, y_max=100.0).to(device)
ATTnet = BoundedNet(input_dim=4, n_hidden_layers=4, n_neurons=64, y_min=0.5, y_max=3.0).to(device)

# Load weights
CBFnet.load_state_dict(torch.load(os.path.join(MODEL_DIR, "CBF_best.pth"), map_location=device))
ATTnet.load_state_dict(torch.load(os.path.join(MODEL_DIR, "ATT_best.pth"), map_location=device))

CBFnet.eval()
ATTnet.eval()

# Load Normalization Stats
X_mean = np.load(os.path.join(MODEL_DIR, "X_mean (1).npy")).astype(np.float32).squeeze()
X_std = np.load(os.path.join(MODEL_DIR, "X_std (1).npy")).astype(np.float32).squeeze()

print("Models and normalization stats loaded successfully!")

Using device: cuda
Models and normalization stats loaded successfully!


In [ ]:
# Cell 5: Batch Inference and Output Generation (ROBUST FILE SEARCH)
import os
import numpy as np
import nibabel as nib
import torch
import matplotlib.pyplot as plt

subjects = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))]
print(f"Found {len(subjects)} subjects. Starting processing...")

for subject in subjects:
    print(f"\nProcessing {subject}...")
    subj_path = os.path.join(DATA_ROOT, subject)

    # Create Subject Output Directory
    subj_out_dir = os.path.join(OUTPUT_ROOT, subject)
    os.makedirs(subj_out_dir, exist_ok=True)

    # ==========================================
    # 1. Robust File Discovery (Subject-Specific)
    # ==========================================
    # We look inside the specific subject's folder (subj_path)
    all_files = os.listdir(subj_path)

    # Find files by checking if specific words are in the filename
    perf_files = sorted([f for f in all_files if 'perfusion' in f.lower() and f.endswith('.nii.gz')])
    m0_files = sorted([f for f in all_files if 'm0' in f.lower() and f.endswith('.nii.gz')])
    mask_files = [f for f in all_files if 'mask' in f.lower() and f.endswith('.nii.gz')]

    # Validation checks
    if len(perf_files) != 4 or len(m0_files) != 4:
        print(f"  -> Skipping {subject}: Found {len(perf_files)} Perfusion and {len(m0_files)} M0 files. Need exactly 4 of each.")
        continue

    if len(mask_files) == 0:
        print(f"  -> Skipping {subject}: Could not find any file containing 'mask' in the name.")
        continue

    mask_file_path = os.path.join(subj_path, mask_files[0])

    # ==========================================
    # 2. Load Data for this Subject
    # ==========================================
    # Load the specific mask for this subject
    mask_img = nib.load(mask_file_path)
    affine = mask_img.affine
    mask_data = mask_img.get_fdata() > 0
    sz = mask_data.shape

    deltaM_vols = np.zeros((*sz, 4), dtype=np.float32)
    M0_vols = np.zeros((*sz, 4), dtype=np.float32)

    # Load the 4 Perfusion and M0 files for this subject
    for j in range(4):
        deltaM_vols[..., j] = nib.load(os.path.join(subj_path, perf_files[j])).get_fdata()
        M0_vols[..., j] = nib.load(os.path.join(subj_path, m0_files[j])).get_fdata()

    # ==========================================
    # 3. Fast Batched Inference
    # ==========================================
    CBF_map = np.full(sz, np.nan, dtype=np.float32)
    ATT_map = np.full(sz, np.nan, dtype=np.float32)

    M0_mean = np.mean(M0_vols, axis=-1).astype(np.float32)

    # Combine the anatomical mask WITH your M0 > 100 threshold
    valid_mask = mask_data & (M0_mean > 100)
    coords = np.argwhere(valid_mask)

    if len(coords) == 0:
        print(f"  -> Skipping {subject}: No valid voxels found passing threshold.")
        continue

    features = np.empty((len(coords), 4), dtype=np.float32)
    for i, (x, y, z) in enumerate(coords):
        dm = deltaM_vols[x, y, z, :].astype(np.float32)
        m0 = M0_mean[x, y, z]
        features[i] = dm / (100.0 * m0)

    features_n = (features - X_mean) / X_std

    with torch.no_grad():
        t_features = torch.tensor(features_n, dtype=torch.float32).to(device)
        cbf_preds = CBFnet(t_features).squeeze().cpu().numpy()
        att_preds = ATTnet(t_features).squeeze().cpu().numpy()

    for i, (x, y, z) in enumerate(coords):
        CBF_map[x, y, z] = cbf_preds[i]
        ATT_map[x, y, z] = att_preds[i]

    # ==========================================
    # 4. Save Output NIfTI Files
    # ==========================================
    CBF_save = np.nan_to_num(CBF_map, nan=0.0)
    ATT_save = np.nan_to_num(ATT_map, nan=0.0)

    nib.save(nib.Nifti1Image(CBF_save, affine), os.path.join(subj_out_dir, f"dnn_cbf_{subject}.nii.gz"))
    nib.save(nib.Nifti1Image(ATT_save, affine), os.path.join(subj_out_dir, f"dnn_att_{subject}.nii.gz"))

    # ==========================================
    # 5. Visualization (MATCHING TARGET FORMAT)
    # ==========================================
    fig, axes = plt.subplots(3, 2, figsize=(10, 14))
    fig.patch.set_facecolor('white')

    # Add the main bold title at the top
    fig.suptitle(f'{subject} Dataset - DNN ASL Results (Slices 15, 20, 25)', fontsize=16, fontweight='bold', y=0.98)

    slices = [15, 20, 25]
    for i, z in enumerate(slices):
        if z >= sz[2]: continue

        # --- CBF Plot ---
        cbf_slice = np.rot90(CBF_map[:, :, z])
        # Mask NaNs and exact zeros to ensure a pure white background
        cbf_slice_masked = np.ma.masked_where(np.isnan(cbf_slice) | (cbf_slice == 0), cbf_slice)

        # Calculate dynamic min/max for the title
        cbf_min = cbf_slice_masked.min() if cbf_slice_masked.count() > 0 else 0.0
        cbf_max = cbf_slice_masked.max() if cbf_slice_masked.count() > 0 else 0.0

        # Plot CBF (vmin=0, vmax=80 matches your reference image)
        im_cbf = axes[i, 0].imshow(cbf_slice_masked, cmap='jet', vmin=0, vmax=80)
        axes[i, 0].set_title(f'CBF Slice {z} ({cbf_min:.1f}-{cbf_max:.1f} ml/100g/min)', color='black', fontsize=11)
        axes[i, 0].axis('off')

        cb_cbf = fig.colorbar(im_cbf, ax=axes[i, 0], fraction=0.046, pad=0.04)
        cb_cbf.ax.yaxis.set_tick_params(color='black', labelcolor='black')
        cb_cbf.outline.set_edgecolor('black')

        # --- ATT Plot ---
        att_slice = np.rot90(ATT_map[:, :, z])
        # Mask NaNs and exact zeros
        att_slice_masked = np.ma.masked_where(np.isnan(att_slice) | (att_slice == 0), att_slice)

        # Calculate dynamic min/max for the title
        att_min = att_slice_masked.min() if att_slice_masked.count() > 0 else 0.0
        att_max = att_slice_masked.max() if att_slice_masked.count() > 0 else 0.0

        im_att = axes[i, 1].imshow(att_slice_masked, cmap='jet', vmin=0.5, vmax=3.0)
        axes[i, 1].set_title(f'ATT Slice {z} ({att_min:.2f}-{att_max:.2f} s)', color='black', fontsize=11)
        axes[i, 1].axis('off')

        cb_att = fig.colorbar(im_att, ax=axes[i, 1], fraction=0.046, pad=0.04)
        cb_att.ax.yaxis.set_tick_params(color='black', labelcolor='black')
        cb_att.outline.set_edgecolor('black')

    # Adjust layout to prevent overlapping text, leaving room for the suptitle
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(subj_out_dir, f"dnn_slices_{subject}.png"), facecolor='white', bbox_inches='tight', dpi=150)
    plt.close()

    print(f"  -> Successfully generated outputs for {subject}.")

print("\n--- ALL PROCESSING COMPLETE ---")

Found 18 subjects. Starting processing...

Processing 27 rajeswari...
  -> Successfully generated outputs for 27 rajeswari.

Processing 29 aby...
  -> Successfully generated outputs for 29 aby.

Processing 36 biji...
  -> Successfully generated outputs for 36 biji.

Processing 34 shenbagavalli...
  -> Successfully generated outputs for 34 shenbagavalli.

Processing 28 krishnaveni...
  -> Successfully generated outputs for 28 krishnaveni.

Processing 33 shakthi...
  -> Successfully generated outputs for 33 shakthi.

Processing 25 divya...
  -> Successfully generated outputs for 25 divya.

Processing 26 shahzaib...
  -> Successfully generated outputs for 26 shahzaib.

Processing 24 pandiyaraj...
  -> Successfully generated outputs for 24 pandiyaraj.

Processing 23 mriganka...
  -> Successfully generated outputs for 23 mriganka.

Processing 20 vedhika...
  -> Successfully generated outputs for 20 vedhika.

Processing 19 sumathi...
  -> Successfully generated outputs for 19 sumathi.

Proce

In [ ]:
# Cell 5: Batch Inference and Output Generation (ROBUST FILE SEARCH)
import os
import numpy as np
import nibabel as nib
import torch
import matplotlib.pyplot as plt

subjects = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))]
print(f"Found {len(subjects)} subjects. Starting processing...")

for subject in subjects:
    print(f"\nProcessing {subject}...")
    subj_path = os.path.join(DATA_ROOT, subject)

    # Create Subject Output Directory
    subj_out_dir = os.path.join(OUTPUT_ROOT, subject)
    os.makedirs(subj_out_dir, exist_ok=True)

    # ==========================================
    # 1. Robust File Discovery (Subject-Specific)
    # ==========================================
    # We look inside the specific subject's folder (subj_path)
    all_files = os.listdir(subj_path)

    # Find files by checking if specific words are in the filename
    perf_files = sorted([f for f in all_files if 'perfusion' in f.lower() and f.endswith('.nii.gz')])
    m0_files = sorted([f for f in all_files if 'm0' in f.lower() and f.endswith('.nii.gz')])
    mask_files = [f for f in all_files if 'mask' in f.lower() and f.endswith('.nii.gz')]

    # Validation checks
    if len(perf_files) != 4 or len(m0_files) != 4:
        print(f"  -> Skipping {subject}: Found {len(perf_files)} Perfusion and {len(m0_files)} M0 files. Need exactly 4 of each.")
        continue

    if len(mask_files) == 0:
        print(f"  -> Skipping {subject}: Could not find any file containing 'mask' in the name.")
        continue

    mask_file_path = os.path.join(subj_path, mask_files[0])

    # ==========================================
    # 2. Load Data for this Subject
    # ==========================================
    # Load the specific mask for this subject
    mask_img = nib.load(mask_file_path)
    affine = mask_img.affine
    mask_data = mask_img.get_fdata() > 0
    sz = mask_data.shape

    deltaM_vols = np.zeros((*sz, 4), dtype=np.float32)
    M0_vols = np.zeros((*sz, 4), dtype=np.float32)

    # Load the 4 Perfusion and M0 files for this subject
    for j in range(4):
        deltaM_vols[..., j] = nib.load(os.path.join(subj_path, perf_files[j])).get_fdata()
        M0_vols[..., j] = nib.load(os.path.join(subj_path, m0_files[j])).get_fdata()

    # ==========================================
    # 3. Fast Batched Inference
    # ==========================================
    CBF_map = np.full(sz, np.nan, dtype=np.float32)
    ATT_map = np.full(sz, np.nan, dtype=np.float32)

    M0_mean = np.mean(M0_vols, axis=-1).astype(np.float32)

    # Combine the anatomical mask WITH your M0 > 100 threshold
    valid_mask = mask_data & (M0_mean > 100)
    coords = np.argwhere(valid_mask)

    if len(coords) == 0:
        print(f"  -> Skipping {subject}: No valid voxels found passing threshold.")
        continue

    features = np.empty((len(coords), 4), dtype=np.float32)
    for i, (x, y, z) in enumerate(coords):
        dm = deltaM_vols[x, y, z, :].astype(np.float32)
        m0 = M0_mean[x, y, z]
        features[i] = dm / (100.0 * m0)

    features_n = (features - X_mean) / X_std

    with torch.no_grad():
        t_features = torch.tensor(features_n, dtype=torch.float32).to(device)
        cbf_preds = CBFnet(t_features).squeeze().cpu().numpy()
        att_preds = ATTnet(t_features).squeeze().cpu().numpy()

    for i, (x, y, z) in enumerate(coords):
        CBF_map[x, y, z] = cbf_preds[i]
        ATT_map[x, y, z] = att_preds[i]

    # ==========================================
    # 4. Save Output NIfTI Files
    # ==========================================
    CBF_save = np.nan_to_num(CBF_map, nan=0.0)
    ATT_save = np.nan_to_num(ATT_map, nan=0.0)

    nib.save(nib.Nifti1Image(CBF_save, affine), os.path.join(subj_out_dir, f"dnn_cbf_{subject}.nii.gz"))
    nib.save(nib.Nifti1Image(ATT_save, affine), os.path.join(subj_out_dir, f"dnn_att_{subject}.nii.gz"))

    # ==========================================
    # 5. Visualization (MATCHING TARGET FORMAT)
    # ==========================================
    fig, axes = plt.subplots(3, 2, figsize=(10, 14))
    fig.patch.set_facecolor('white')

    # Add the main bold title at the top
    fig.suptitle(f'DNN ASL Results — Subject: {subject}\nCBF (ml/100g/min) and ATT (s)', fontsize=16, fontweight='bold', y=0.98)

    # Calculate global mean and median for the entire valid brain volume
    global_cbf_mean = np.nanmean(CBF_map[valid_mask])
    global_cbf_median = np.nanmedian(CBF_map[valid_mask])
    global_att_mean = np.nanmean(ATT_map[valid_mask])
    global_att_median = np.nanmedian(ATT_map[valid_mask])


    slices = [15, 20, 25]
    for i, z in enumerate(slices):
        if z >= sz[2]: continue

        # --- CBF Plot ---
        cbf_slice = np.rot90(CBF_map[:, :, z])
        # Mask NaNs and exact zeros to ensure a pure white background
        cbf_slice_masked = np.ma.masked_where(np.isnan(cbf_slice) | (cbf_slice == 0), cbf_slice)

        # Calculate dynamic min/max for the title
        cbf_min = cbf_slice_masked.min() if cbf_slice_masked.count() > 0 else 0.0
        cbf_max = cbf_slice_masked.max() if cbf_slice_masked.count() > 0 else 0.0

        # Plot CBF (vmin=0, vmax=80 matches your reference image)
        im_cbf = axes[i, 0].imshow(cbf_slice_masked, cmap='jet', vmin=0, vmax=80)
        axes[i, 0].set_title(f'CBF Slice {z} ({cbf_min:.1f}-{cbf_max:.1f} ml/100g/min)\nmean={global_cbf_mean:.1f} median={global_cbf_median:.1f}', color='black', fontsize=11)
        axes[i, 0].axis('off')

        cb_cbf = fig.colorbar(im_cbf, ax=axes[i, 0], fraction=0.046, pad=0.04)
        cb_cbf.ax.yaxis.set_tick_params(color='black', labelcolor='black')
        cb_cbf.outline.set_edgecolor('black')

        # --- ATT Plot ---
        att_slice = np.rot90(ATT_map[:, :, z])
        # Mask NaNs and exact zeros
        att_slice_masked = np.ma.masked_where(np.isnan(att_slice) | (att_slice == 0), att_slice)

        # Calculate dynamic min/max for the title
        att_min = att_slice_masked.min() if att_slice_masked.count() > 0 else 0.0
        att_max = att_slice_masked.max() if att_slice_masked.count() > 0 else 0.0

        im_att = axes[i, 1].imshow(att_slice_masked, cmap='jet', vmin=0.5, vmax=3.0)
        axes[i, 1].set_title(f'ATT Slice {z} ({att_min:.2f}-{att_max:.2f} s)\nmean={global_att_mean:.2f} median={global_att_median:.2f}', color='black', fontsize=11)
        axes[i, 1].axis('off')

        cb_att = fig.colorbar(im_att, ax=axes[i, 1], fraction=0.046, pad=0.04)
        cb_att.ax.yaxis.set_tick_params(color='black', labelcolor='black')
        cb_att.outline.set_edgecolor('black')

    # Adjust layout to prevent overlapping text, leaving room for the suptitle
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(subj_out_dir, f"dnn_slices_{subject}.png"), facecolor='white', bbox_inches='tight', dpi=150)
    plt.close()

    print(f"  -> Successfully generated outputs for {subject}.")

print("\n--- ALL PROCESSING COMPLETE ---")

Found 18 subjects. Starting processing...

Processing 27 rajeswari...
  -> Successfully generated outputs for 27 rajeswari.

Processing 29 aby...
  -> Successfully generated outputs for 29 aby.

Processing 36 biji...
  -> Successfully generated outputs for 36 biji.

Processing 34 shenbagavalli...
  -> Successfully generated outputs for 34 shenbagavalli.

Processing 28 krishnaveni...
  -> Successfully generated outputs for 28 krishnaveni.

Processing 33 shakthi...
  -> Successfully generated outputs for 33 shakthi.

Processing 25 divya...
  -> Successfully generated outputs for 25 divya.

Processing 26 shahzaib...
  -> Successfully generated outputs for 26 shahzaib.

Processing 24 pandiyaraj...
  -> Successfully generated outputs for 24 pandiyaraj.

Processing 23 mriganka...
  -> Successfully generated outputs for 23 mriganka.

Processing 20 vedhika...
  -> Successfully generated outputs for 20 vedhika.

Processing 19 sumathi...
  -> Successfully generated outputs for 19 sumathi.

Proce